# Financial Data Lakehouse

This interactive notebook operates as the validation and analytical console.

## Section 0: DuckDB Initialization & Wildcard View Registrations
Initializes in-process DuckDB instance and registers unified zero-copy views across historical (1999~2023) and incremental (2024~present) Parquet tables.

In [16]:
import json
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display, HTML
from data_scripts import portfolio_calc, viz_render

con = duckdb.connect()

# Register wildcard views across Lakehouse domains
con.execute("CREATE VIEW v_prices AS SELECT * FROM read_parquet('yfinance/prices_*.parquet', union_by_name = true)")
con.execute("CREATE VIEW v_indices AS SELECT * FROM read_parquet('yfinance/indices.parquet')")
con.execute("CREATE VIEW v_constituents AS SELECT * FROM read_parquet('constituents_*.parquet', union_by_name = true)")
con.execute("CREATE VIEW v_macro AS SELECT * FROM read_parquet('fred/indicators.parquet')")
con.execute("CREATE VIEW v_news AS SELECT * FROM read_parquet('news/*.parquet', union_by_name = true)")

print("DuckDB zero-copy views initialized successfully:")
print(f"  - v_prices: {con.execute('SELECT COUNT(*) FROM v_prices').fetchone()[0]:,} rows")
print(f"  - v_indices: {con.execute('SELECT COUNT(*) FROM v_indices').fetchone()[0]:,} rows")
print(f"  - v_constituents: {con.execute('SELECT COUNT(*) FROM v_constituents').fetchone()[0]:,} rows")
print(f"  - v_macro: {con.execute('SELECT COUNT(*) FROM v_macro').fetchone()[0]:,} rows")
print(f"  - v_news: {con.execute('SELECT COUNT(*) FROM v_news').fetchone()[0]:,} rows")

DuckDB zero-copy views initialized successfully:
  - v_prices: 1,037,733 rows
  - v_constituents: 5,050 rows
  - v_macro: 80,068 rows
  - v_news: 882,831 rows


## View 1: Data Lakehouse Health Console (System Integrity)
Audits synchronization timestamps, partition records, null rates, and local disk footprints.

In [17]:
with open("sync_state.json", "r", encoding="utf-8") as f:
    telemetry = json.load(f)

print(f"Last Global Synchronized (UTC): {telemetry.get('last_sync_timestamp_utc')}")
df_health = viz_render.render_lakehouse_health(telemetry)
display(df_health)

Last Global Synchronized (UTC): 2026-10-05T12:16:30.267483+00:00


,Lakehouse Domain,Total Rows,Entity/Series Count,Min Date,Max Date,Health Status
0,CONSTITUENTS,"5,050",187,1999-03-31,2026-10-05,HEALTHY
1,PRICES,"1,031,444",193,1999-01-04,2026-10-02,HEALTHY
2,MACRO_INDICATORS,"80,068",28,1854-12-01,2026-10-02,HEALTHY
3,NEWS,"882,831",-,2001-05-31,2026-10-05,HEALTHY


## View 2: Macro Regime & Top-Down News View (Macro Context)
Cross-references equity index movements (SPY, 1999~2026) with Federal Reserve monetary policy cycles, audits lakehouse news distribution across categories, and provides an interactive Top-Down news reader console.

In [18]:
# 2.1 Macro Regime Overlay (SPY Benchmark vs Federal Funds Rate 1999~2026)
df_prices = con.execute("SELECT date, ticker, adj_close FROM v_prices WHERE ticker = 'SPY'").df()
df_macro = con.execute("SELECT date, series_id, value FROM v_macro WHERE series_id = 'FEDFUNDS' AND date >= '1999-01-01'").df()

fig_macro = viz_render.render_macro_overlay(
    df_prices=df_prices,
    df_macro=df_macro,
    benchmark_ticker="SPY",
    macro_series_id="FEDFUNDS",
    macro_series_name="Federal Funds Effective Rate (%)"
)
fig_macro.show()

# 2.2 Lakehouse Top-Down News Universe Breakdown
print("LAKEHOUSE TOP-DOWN NEWS DOMAIN BREAKDOWN:")
df_news_dist = con.execute("""
    SELECT 
        category,
        COUNT(*) AS total_articles,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_share,
        MIN(timestamp)::DATE AS earliest_date,
        MAX(timestamp)::DATE AS latest_date
    FROM v_news
    GROUP BY category
    ORDER BY total_articles DESC
""").df()
display(df_news_dist)

LAKEHOUSE TOP-DOWN NEWS DOMAIN BREAKDOWN:


,category,total_articles,pct_share,earliest_date,latest_date
0,market,526349,59.6,2008-11-17,2026-10-05
1,macro,306026,34.7,2001-05-31,2026-10-05
2,sector,50456,5.7,2009-07-27,2026-10-05


### Top-Down News Reader & Macro Regime Console
Extracts and displays readable, non-truncated news articles across distinct macroeconomic regimes (e.g. 2024~2026 Policy Pivot, 2022 Inflation Shock, 2008 GFC) or user-specified macro keywords.

In [19]:
def query_top_down_news(category="macro", keyword=None, date_start=None, date_end=None, limit=10):
    """Interactive top-down query extracting readable news records from DuckDB."""
    conditions = []
    if category:
        conditions.append(f"category = '{category}'")
    if keyword:
        kw = keyword.lower()
        conditions.append(f"(LOWER(headline) LIKE '%{kw}%' OR LOWER(summary) LIKE '%{kw}%')")
    if date_start:
        conditions.append(f"timestamp >= '{date_start}'")
    if date_end:
        conditions.append(f"timestamp <= '{date_end}'")
    
    where_clause = " WHERE " + " AND ".join(conditions) if conditions else ""
    query = f"""
        SELECT timestamp, source, category, headline, summary
        FROM v_news
        {where_clause}
        ORDER BY timestamp DESC
        LIMIT {limit}
    """
    df = con.execute(query).df()
    return HTML(viz_render.render_news_feed_html(df, limit=limit))

# Regime 1: 2024~2026 Fed Policy Pivot & Rate Decisions
print("=== [Regime 1: 2024~2026 Fed Policy Pivot & Rate Decisions] ===")
display(query_top_down_news(category="macro", keyword="fed", date_start="2024-01-01", limit=5))

# Regime 2: 2022~2023 Inflation Shock & Aggressive Tightening
print("=== [Regime 2: 2022~2023 Inflation Shock & Aggressive Tightening] ===")
display(query_top_down_news(category="macro", keyword="inflation", date_start="2022-01-01", date_end="2022-12-31", limit=4))

# Regime 3: 2008 Global Financial Crisis Interventions
print("=== [Regime 3: 2008 Global Financial Crisis Interventions] ===")
display(query_top_down_news(category="macro", keyword="bailout", date_start="2008-09-01", date_end="2008-12-31", limit=3))

=== [Regime 1: 2024~2026 Fed Policy Pivot & Rate Decisions] ===


Timestamp,Category,Source,Headline & Summary
2026-10-05 10:03,macro,Google News Macro,Can holiday spending lift Oklahoma's economy? What the Fed is watching - The Oklahoman Can holiday spending lift Oklahoma's economy? What the Fed is watching The Oklahoman
2026-10-05 08:25,macro,Google News Macro,Trump Adviser Calls for Powell Exit as Bitcoin Watches Fed’s Next Move - Bitcoin Foundation Trump Adviser Calls for Powell Exit as Bitcoin Watches Fed’s Next Move Bitcoin Foundation
2026-10-05 06:58,macro,Google News Macro,Tokyo shares are higher as easing worries over inflation reduce odds for another Fed rate hike - KSAT Tokyo shares are higher as easing worries over inflation reduce odds for another Fed rate hike KSAT
2026-10-05 06:24,macro,Google News Macro,"The Fed Builds the GENIUS Act Bank-Grade Plumbing — Two NPRMs, Two Paths, One Deadline - forkast.news The Fed Builds the GENIUS Act Bank-Grade Plumbing — Two NPRMs, Two Paths, One Deadline forkast.news"
2026-10-05 04:20,macro,Google News Macro,Asian shares are higher as easing worries over inflation reduce odds for another Fed rate hike - WTOP News Asian shares are higher as easing worries over inflation reduce odds for another Fed rate hike WTOP News


=== [Regime 2: 2022~2023 Inflation Shock & Aggressive Tightening] ===


Timestamp,Category,Source,Headline & Summary
2022-12-30 00:00,macro,Unknown,"High Demand, Online Strength Aid Aaron's (AAN) Amid Inflation The company’s latest acquisition of appliance and electronics retailer, BrandsMart, enabled it to strengthen its market position and expand the customer base. Notably, revenues were $183.3 million in the third quarter of 2022, driven by strength in appliances and e-commerce, which offset weak product sales from the deflation in certain consumer electronics categories. Conclusion We believe that a solid online show, the BrandsMart buyout and continued strength in the GenNext strategy will aid Aaron’s and help offset inflationary pressures."
2022-12-30 00:00,macro,Unknown,"GRAPHIC-Inflation, recession and earnings among factors to drive U.S. stocks in 2023 High-profile casualties include the once-soaring shares of Amazon.com Inc AMZN.O, which have slumped around 50% this year, while those of Tesla Inc TSLA.O and Facebook parent Meta Platforms Inc META.O have both lost about 65%. The Fed’s rate hikes have pushed up bond yields and created competition for equities, flying in the face of the low-yield environment that predominated for more than a decade and gave rise to the acronym “TINA,” or “there is no alternative” to stocks. The greenback has pared some of those gains in recent weeks and a continued reversal would depend in part on investor perceptions of how hawkish the Fed will be relative to other global central banks."
2022-12-29 00:00,macro,Unknown,"3 Energy Pipeline Stocks for Strong Yield, Inflation Hedge As the above scenario pans out with upside inflation surprises, investing in high-quality energy infrastructure stocks like DCP Midstream Partners DCP, Global Partners LP GLP and Delek Logistics Partners DKL might help you earn a decent return. DKL pays out 99 cents quarterly distribution ($3.96 per unit annually), which gives it an 8.39% yield at the current unit price. Valued at around $2 billion, DKL, a Zacks Rank #2 (Buy) stock, has gained 10% in 2022."
2022-12-29 00:00,macro,Unknown,"U.S. Treasury says consumer leases can qualify for EV tax credits That law lifts the 200,000-vehicle per manufacturer cap that had made Tesla TSLA.O and General Motors GM.N ineligible for EV tax credits starting Jan. 1. The $430 billion U.S. Inflation Reduction Act (IRA) passed in August ended $7,500 consumer tax credits for purchases of electric vehicles assembled outside North America, angering South Korea, the European Union, Japan and others. Senator Joe Manchin, a Democrat who chairs the chamber's energy panel, had urged Treasury to reject allowing use of the credit in consumer leasing saying they were trying to use the provision ""as a way to bypass the strict sourcing requirements."""


=== [Regime 3: 2008 Global Financial Crisis Interventions] ===


Timestamp,Category,Source,Headline & Summary
2008-12-19 17:55,macro,Unknown,Jobs with Justice and Teamsters Union Call for 'People's Bailout' at KeyBank Branches... Jobs with Justice and Teamsters Union Call for 'People's Bailout' at KeyBank Branches...
2008-12-19 02:44,macro,Unknown,Treasury's Paulson vague on auto bailout prospects Treasury's Paulson vague on auto bailout prospects
2008-12-19 02:39,macro,Unknown,UPDATE 1-Treasury's Paulson vague on auto bailout prospects UPDATE 1-Treasury's Paulson vague on auto bailout prospects


## View 3: Constituent Churn & Weight Matrix View (Portfolio Composition)
Visualizes S&P 500 Top 50 capital concentration via an interactive treemap and inspects quarterly ingress/egress churn.

In [20]:
df_constituents = con.execute("SELECT date, ticker, market_cap, weight FROM v_constituents").df()

# Latest Top 50 Weight Allocation Treemap
fig_treemap = viz_render.render_weights_treemap(df_constituents)
fig_treemap.show()

# Member Ingress/Egress Churn Table between recent periods
df_churn = viz_render.render_churn_table(df_constituents)
print(f"Rebalancing Churn Matrix (Recent Rebalancing Changes):")
display(df_churn.head(10))

Rebalancing Churn Matrix (Recent Rebalancing Changes):


,ticker,Weight (2023-12-31),Weight (2026-10-05),Delta (%),Status
0,NVDA,0.0,11.36,11.36,NEW INGRESS (ADD)
1,GOOGL,0.0,8.45,8.45,NEW INGRESS (ADD)
2,META,0.0,3.73,3.73,NEW INGRESS (ADD)
3,BRK-B,0.0,2.16,2.16,NEW INGRESS (ADD)
4,MA,0.0,0.97,0.97,NEW INGRESS (ADD)
5,PLTR,0.0,0.91,0.91,NEW INGRESS (ADD)
6,CSCO,0.0,0.89,0.89,NEW INGRESS (ADD)
7,BAC,0.0,0.76,0.76,NEW INGRESS (ADD)
8,DELL,0.0,0.72,0.72,NEW INGRESS (ADD)
9,PG,0.0,0.68,0.68,NEW INGRESS (ADD)


## View 4: Top 50 Value-Weighted Performance View (Strategy Evaluation)
Computes historical portfolio returns from normalized capital weights and evaluates risk-adjusted metrics against SPY.

In [21]:
df_all_prices = con.execute("SELECT date, ticker, adj_close FROM v_prices").df()
df_weights = portfolio_calc.compute_weights(df_constituents)

df_returns = portfolio_calc.compute_portfolio_returns(df_all_prices, df_weights, benchmark_ticker="SPY")
metrics = portfolio_calc.compute_performance_metrics(df_returns)

# Print Summary Scorecard
print("=" * 65)
print("STRATEGY SCORECARD: S&P 500 Top 50 Value-Weighted vs SPY")
print("=" * 65)
print(f"Total Trading Sessions : {metrics['n_trading_days']:,} days")
print(f"Cumulative Total Return: Strategy {metrics['strategy_total_return']*100:.2f}% | SPY {metrics.get('benchmark_total_return', 0)*100:.2f}%")
print(f"Compound Annual Return : Strategy {metrics['strategy_cagr']*100:.2f}% | SPY {metrics.get('benchmark_cagr', 0)*100:.2f}%")
print(f"Annualized Volatility  : Strategy {metrics['strategy_annualized_vol']*100:.2f}% | SPY {metrics.get('benchmark_annualized_vol', 0)*100:.2f}%")
print(f"Maximum Drawdown (MDD) : Strategy {metrics['strategy_mdd']*100:.2f}% | SPY {metrics.get('benchmark_mdd', 0)*100:.2f}%")
print(f"Sharpe Ratio (Rf=2%)   : Strategy {metrics['strategy_sharpe']:.2f} | SPY {metrics.get('benchmark_sharpe', 0):.2f}")
print(f"Alpha vs Benchmark     : {metrics.get('alpha', 0)*100:.2f}%")
print(f"Beta vs Benchmark      : {metrics.get('beta', 0):.2f}")
print("=" * 65)

# Dual-panel interactive Plotly figure
fig_perf = viz_render.render_performance_dashboard(metrics["detailed_df"])
fig_perf.show()

STRATEGY SCORECARD: S&P 500 Top 50 Value-Weighted vs SPY
Total Trading Sessions : 6,919 days
Cumulative Total Return: Strategy 907.16% | SPY 854.36%
Compound Annual Return : Strategy 8.78% | SPY 8.56%
Annualized Volatility  : Strategy 25.80% | SPY 19.19%
Maximum Drawdown (MDD) : Strategy -70.36% | SPY -55.19%
Sharpe Ratio (Rf=2%)   : Strategy 0.26 | SPY 0.34
Alpha vs Benchmark     : -1.14%
Beta vs Benchmark      : 1.21
